# ME 415 — Homework 1, Problem 1b

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byuflowlab/flightlab/blob/main/notebooks/hw1_starter.ipynb)

## Constrained pod-shape study

This notebook is only for **Problem 1b**. Complete Problem 1a entirely in the FlightLab Workbench, then save its canonical project as `rc1-hw1.flightlab.json`. Problems 2a and 2b are independent calculations and are not part of this notebook.

Replace every **TODO** and `np.nan` placeholder.

## Colab workflow

1. Sign into a Google account and open this notebook in Colab.
2. Choose **File → Save a copy in Drive** so your changes are saved.
3. Run the cells below in order.
4. When prompted, upload the `rc1-hw1.flightlab.json` file created in the workbench.
5. Download or save your completed notebook before submitting it. A Colab runtime and its uploaded files are temporary.

If you are running the notebook locally instead, put `rc1-hw1.flightlab.json` in the same folder as the notebook.

## 0. Set up FlightLab

Run the next cell whenever you start a new Colab runtime. It checks for the current tested course build and installs it in Colab's temporary Python environment.

In [ ]:
import re
import subprocess
import sys
from urllib.request import urlopen

_fallback_commit = "0ee06b60ba2d657cb7dbe324faef81d2c8be8e5a"
_release_url = (
    "https://raw.githubusercontent.com/byuflowlab/flightlab/"
    "main/student_setup/release.txt"
)
try:
    _course_commit = urlopen(_release_url, timeout=10).read().decode().strip()
    if re.fullmatch(r"[0-9a-f]{40}", _course_commit) is None:
        raise ValueError("invalid course release")
except (OSError, UnicodeError, ValueError):
    _course_commit = _fallback_commit
    print("Update check unavailable; using the notebook's included course build.")

_requirement = (
    "flightlab @ https://github.com/byuflowlab/flightlab/archive/"
    f"{_course_commit}.zip"
)
print(f"Installing FlightLab course build {_course_commit[:8]}...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", _requirement])

In [ ]:
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import flightlab
from flightlab.project import AircraftProject
from flightlab.project_analysis import run_design_point

print(f"FlightLab {flightlab.__version__} is ready.")

## 1. Load the project from Problem 1a

In Colab, the next cell opens an upload chooser. Upload the `rc1-hw1.flightlab.json` file you saved from the workbench. The notebook does not build or replace the Problem 1a model.

In [ ]:
try:
    from google.colab import files
except ImportError:
    files = None

if files is not None:
    uploaded = files.upload()
    project_files = [name for name in uploaded if name.endswith(".flightlab.json")]
    if len(project_files) != 1:
        raise ValueError("Upload exactly one .flightlab.json project file.")
    project_filename = project_files[0]
    project = AircraftProject.from_json(uploaded[project_filename].decode("utf-8"))
else:
    project_filename = "rc1-hw1.flightlab.json"
    project_path = Path(project_filename)
    if not project_path.exists():
        raise FileNotFoundError(
            f"Put {project_filename} in the notebook's current folder, then rerun this cell."
        )
    project = AircraftProject.load(project_path)

print(f"Loaded {project_filename}: {project.name}")

In [ ]:
project.require_valid()
if project.body_named("Fuselage pod") is None:
    raise ValueError("The project must contain a body named 'Fuselage pod'.")
project.case("Cruise")
print("The project contains the pod and Cruise case needed for Problem 1b.")

## 2. Sweep the pod shape

For every trial length, preserve the baseline volume and width-to-height ratio:

$$V=Lwh=0.00440\;\mathrm{m^3}, \qquad r=w/h=1.10.$$

Derive expressions for $h(L)$ and $w(L)$ before filling in the two marked lines. Make a fresh copy of the baseline project for every trial so changes do not accumulate.

In [ ]:
baseline = deepcopy(project)
volume = 0.400 * 0.110 * 0.100  # m^3
width_to_height = 1.10
lengths = np.linspace(0.250, 0.695, 31)

widths = []
heights = []
drag = []

for length in lengths:
    candidate = deepcopy(baseline)
    pod = candidate.body_named("Fuselage pod")

    # TODO: translate the two constraint equations into Python.
    height = np.nan
    width = np.nan
    if not (np.isfinite(height) and np.isfinite(width)):
        raise NotImplementedError("Replace height and width with your constraint equations.")

    pod.length = float(length)
    pod.width = float(width)
    pod.height = float(height)
    candidate.require_valid()

    result = run_design_point(candidate, candidate.case("Cruise"), ns=28, nc=4)
    widths.append(width)
    heights.append(height)
    drag.append(result.drag)

drag = np.asarray(drag)
print(f"Completed {len(drag)} Cruise design points.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(lengths, drag, "o-", markersize=4)
ax.set_xlabel("pod length [m]")
ax.set_ylabel("total aircraft drag [N]")
ax.set_title("RC-1 constrained pod-shape study — Cruise")
ax.grid(True, alpha=0.3)
plt.show()

**Interpretation:** Briefly describe the drag trend shown in your plot. Explain the geometric or packaging reason that the sweep stops at 0.695 m.

## Submission check

Before submitting, confirm that:

- every `np.nan`, **TODO**, and response prompt has been replaced;
- the notebook runs in order from a fresh runtime after uploading your project;
- the pod-sweep plot has labeled axes and units; and
- you have saved or downloaded the completed notebook.